# QUBO and Annealing Pipeline

This notebook starts from the terminal subgraphs saved by the previous stage (either `CSP_showcase` or `mwis_pipeline`). It builds QUBOs and runs D-Wave simulated annealing as the standard final solver step. It does not rebuild the MWIS conflict graph or the decomposition.

Quantum annealing on a real QPU uses the same QUBO artifacts, but depends on hardware access and credentials.

In [ ]:
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS pipeline)
CONFIG_PATH = RESULTS_DIR / "_config.json"
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: any stage that produced terminal subgraphs (MWIS decomposition, CSP showcase, ...)
INPUT_RUN_DIR = RESULTS_DIR / run_config["qubo_input_run"]
TERMINAL_SUBGRAPHS_JSON = INPUT_RUN_DIR / "terminal_subgraphs.json"
if not TERMINAL_SUBGRAPHS_JSON.exists():
    raise FileNotFoundError(f"No terminal_subgraphs.json in {INPUT_RUN_DIR}. Run the producing stage first.")

# output: named after the input run, so QUBOs from different producers stay side by side
OUTPUT_RUN_DIR = RESULTS_DIR / "runs" / "qubo_annealing" / INPUT_RUN_DIR.name
OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

# simulated-annealing settings
RUN_SIMULATED_ANNEALING = True
NUM_READS = run_config["num_reads"]
BASE_SEED = run_config["base_seed"]

# loggin environment configuration
(OUTPUT_RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

# terminal subgraphs produced by the decomposition stage
terminal_subgraphs = json.loads(TERMINAL_SUBGRAPHS_JSON.read_text(encoding="utf-8"))

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"Loaded terminal subgraphs from: {INPUT_RUN_DIR.resolve()}")
print(f"Artifacts will be saved to: {OUTPUT_RUN_DIR.resolve()}")
print({"terminal_subgraphs": len(terminal_subgraphs)})

## Build terminal QUBOs

In [ ]:
def graph_from_terminal(record: dict[str, Any]) -> nx.Graph:
    """
    Rebuild one terminal subgraph from its JSON record.
    """
    graph = nx.Graph()
    # re-building nodes
    for node in record["graph"]["nodes"]:
        node = dict(node)
        node_id = node.pop("id")          # "id" is the key, the rest are attributes (weight, times, ...)
        graph.add_node(node_id, **node)
    # re-building edges 
    for edge in record["graph"]["edges"]:
        edge = dict(edge)
        source = edge.pop("source")
        target = edge.pop("target")
        graph.add_edge(source, target, **edge)   # keeps the conflict "reason" attribute
    return graph


def default_penalty(graph: nx.Graph) -> float:
    """
    Compute the penalty for a conflicting pair: above any single reward, so a conflict never pays off.
    """
    max_weight = max((data["weight"] for _, data in graph.nodes(data=True)), default=1.0)
    return max_weight + 1.0


def build_qubo_from_graph(graph: nx.Graph, penalty: float | None = None) -> dict[str, Any]:
    """
    Build QUBO instance startng from the given graph.
    """
    if penalty is None:
        penalty = default_penalty(graph)

    # linear reward
    linear = {node: -float(data["weight"]) for node, data in graph.nodes(data=True)}

    # quadratic penalty
    quadratic = {(u, v): float(penalty) for u, v in graph.edges}

    return {
        "variables": sorted(graph.nodes),
        "linear": linear,
        "quadratic": quadratic,
        "penalty": float(penalty),
    }


def qubo_energy(qubo: dict[str, Any], sample: dict[str, int]) -> float:
    """
    Compute the energy of the QUBO assignment.
    """
    energy = sum(coef * sample.get(node, 0) for node, coef in qubo["linear"].items())
    energy += sum(coef * sample.get(u, 0) * sample.get(v, 0) for (u, v), coef in qubo["quadratic"].items())
    return energy


qubo_records = []   # for each temrinal graph build a separate QUBO formulation
term_rows = []      # flat list of every QUBO coefficient
for record in terminal_subgraphs:
    graph = graph_from_terminal(record)
    qubo = build_qubo_from_graph(graph)
    qubo_records.append({"terminal": record, "graph": graph, "qubo": qubo})

    for node, coef in qubo["linear"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "linear", "u": node, "v": "", "coefficient": coef})
    for (u, v), coef in qubo["quadratic"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "quadratic", "u": u, "v": v, "coefficient": coef})

# print out translation summary
qubo_summary = pd.DataFrame([
    {
        "terminal_id": item["terminal"]["terminal_id"],
        "original_component_id": item["terminal"]["original_component_id"],
        "variables": item["graph"].number_of_nodes(),
        "couplers": item["graph"].number_of_edges(),
        "penalty": item["qubo"]["penalty"],
    }
    for item in qubo_records
])
qubo_terms = pd.DataFrame(term_rows)

qubo_summary.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_summary.csv", index=False)
qubo_terms.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_terms.csv", index=False)
qubo_summary.head(10)

,terminal_id,original_component_id,variables,couplers,penalty
0,component_001_dnode_0003,component_001,15,61,80.933
1,component_001_dnode_0004,component_001,12,57,92.271
2,component_001_dnode_0005,component_001,4,6,68.424
3,component_001_dnode_0006,component_001,5,8,92.271
4,component_001_dnode_0007,component_001,2,1,68.424
5,component_001_dnode_0008,component_001,12,49,79.732
6,component_001_dnode_0009,component_001,11,55,92.271
7,component_001_dnode_0010,component_001,4,6,92.271
8,component_001_dnode_0011,component_001,25,87,84.354
9,component_001_dnode_0012,component_001,25,105,77.342


## D-Wave simulated annealing

In [ ]:
def selected_nodes_from_sample(sample: dict[str, int]) -> set[str]:
    """Slots the annealer set to 1, i.e. the ones it scheduled."""
    return {node for node, value in sample.items() if int(value) == 1}


def is_independent(graph: nx.Graph, selected: set[str]) -> bool:
    """Feasibility check: no two selected slots may conflict (penalty high enough)."""
    return all(not graph.has_edge(u, v) for u, v in combinations(selected, 2))


try:
    from dwave.samplers import SimulatedAnnealingSampler
except ImportError as exc:
    raise ModuleNotFoundError(
        "D-Wave simulated annealing is required for this notebook. "
        "Install `dwave-samplers` in the Python 3.13.14 environment, "
        "then restart the kernel and rerun the notebook."
    ) from exc

annealing_rows = []
sampler = SimulatedAnnealingSampler()   # classical stand-in for the QPU, same QUBO interface
for index, item in enumerate(qubo_records):
    qubo = item["qubo"]
    graph = item["graph"]
    terminal_id = item["terminal"]["terminal_id"]

    # The sampler wants one dict: diagonal keys (i, i) carry the linear terms.
    Q = {(node, node): coef for node, coef in qubo["linear"].items()}
    Q.update(qubo["quadratic"])

    # simulate quantum annealing
    response = sampler.sample_qubo(Q, num_reads=NUM_READS, seed=BASE_SEED + index)
    best = response.first                # lowest-energy read out of NUM_READS

    # convert the output into interpretable solution
    sample = {variable: int(best.sample[variable]) for variable in qubo["variables"]}
    selected = selected_nodes_from_sample(sample)
    objective = sum(graph.nodes[node]["weight"] for node in selected)   # compare with the exact solver
    annealing_rows.append({
        "terminal_id": terminal_id,
        "selected_node_count": len(selected),
        "objective": objective,
        "energy": float(best.energy),
        "recomputed_energy": qubo_energy(qubo, sample),
        "is_independent": is_independent(graph, selected),
        "num_reads": NUM_READS,
        "seed": BASE_SEED + index,
    })

pd.DataFrame(annealing_rows).to_csv(OUTPUT_RUN_DIR / "annealing_results.csv", index=False)
print({"annealed_terminals": len(annealing_rows), "output": str(OUTPUT_RUN_DIR / "annealing_results.csv")})

{'annealed_terminals': 72, 'output': 'C:\\Users\\felip\\Documents\\Polimi\\Semester 3\\SE2 Quantum Satisfiability Project\\satelite-optimization-quantum-annealing\\PoliMi-SWE2-research-project\\results\\runs\\mwis_pipeline\\latest_mwis_run\\qubo\\annealing_results.csv'}
